# Load the Hard Hat Analytics synthetic sample
Attach a NEW schema-enabled lakehouse. Upload data/ files to Files/hardhat_sample. Existing sample tables are never overwritten.


In [ ]:
from decimal import Decimal
import csv
from datetime import date
import json
from pathlib import Path
from pyspark.sql.types import (
    StructType, StructField, StringType, DateType, LongType,
    DoubleType, DecimalType, BooleanType,
)

# Upload the repository's data folder to Files/hardhat_sample in a NEW lakehouse.
# Attach that lakehouse as this notebook's default before running.
root = Path("/lakehouse/default/Files/hardhat_sample")
manifest = json.loads((root / "manifest.json").read_text())
frames = {}
types = {
    "varchar": StringType(), "date": DateType(), "bigint": LongType(),
    "int": LongType(), "float": DoubleType(), "bit": BooleanType(),
    "decimal": DecimalType(38, 2),
}

for name, spec in manifest["tables"].items():
    if spark.catalog.tableExists("dbo." + name):
        raise RuntimeError("Table already exists: " + name +
                           ". Use a new lakehouse; this sample never overwrites.")
    fields = [StructField(c["name"], types[c["type"]], True)
              for c in spec["columns"]]
    records = []
    with (root / (name + ".csv")).open(newline="", encoding="utf-8") as handle:
        for row in csv.DictReader(handle):
            record = []
            for column in spec["columns"]:
                value = row[column["name"]]
                dtype = column["type"]
                if value == "":
                    parsed = None
                elif dtype == "date":
                    parsed = date.fromisoformat(value)
                elif dtype in ("bigint", "int"):
                    parsed = int(value)
                elif dtype == "float":
                    parsed = float(value)
                elif dtype == "decimal":
                    parsed = Decimal(value)
                elif dtype == "bit":
                    if value not in ("true", "false"):
                        raise ValueError("Invalid boolean in " + name)
                    parsed = value == "true"
                else:
                    parsed = value
                record.append(parsed)
            records.append(tuple(record))
    if len(records) != spec["rows"]:
        raise ValueError("Unexpected row count: " + name)
    frames[name] = spark.createDataFrame(records, StructType(fields))

# Parse and check all inputs before the first write; preserve other tables.
for name, frame in frames.items():
    frame.write.format("delta").mode("errorifexists").saveAsTable("dbo." + name)
    print(name, frame.count())

display(spark.table("dbo.project_reviews")
        .select("project_id", "project_name", "cost_overrun_amount",
                "recorded_project_delay_days", "attention_status"))
